# Project Introduction

Book Recommendation System using Collaborative Filtering
This notebook implements an Item-Based Collaborative Filtering system. It uses the k-Nearest Neighbors (k-NN) algorithm with cosine similarity to suggest books based on user rating patterns.

Key Stages:

- Data Cleansing: Filtering out inactive users and unpopular books to reduce noise.

- Matrix Transformation: Converting the data into a Sparse Matrix for memory efficiency.

- Model Training: Using an unsupervised Learner to find similarities in high-dimensional space.

# 1. Libraries and Data Setup

In [ ]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import kagglehub

print("Downloading data...")
path = kagglehub.dataset_download("arashnic/book-recommendation-dataset")
print(f"Data downloaded to: {path}")

Using Colab cache for faster access to the 'book-recommendation-dataset' dataset.
Data downloaded to: /kaggle/input/book-recommendation-dataset


# 2. Data Loading and Cleaning

In [ ]:
books = pd.read_csv(f"{path}/Books.csv", on_bad_lines='skip', low_memory=False)
ratings = pd.read_csv(f"{path}/Ratings.csv", on_bad_lines='skip', low_memory=False)


In [ ]:
books

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...
...,...,...,...,...,...,...,...,...
271355,0440400988,There's a Bat in Bunk Five,Paula Danziger,1988,Random House Childrens Pub (Mm),http://images.amazon.com/images/P/0440400988.0...,http://images.amazon.com/images/P/0440400988.0...,http://images.amazon.com/images/P/0440400988.0...
271356,0525447644,From One to One Hundred,Teri Sloat,1991,Dutton Books,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...
271357,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker,2004,HarperSanFrancisco,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...
271358,0192126040,Republic (World's Classics),Plato,1996,Oxford University Press,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...


In [ ]:
ratings

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6
...,...,...,...
1149775,276704,1563526298,9
1149776,276706,0679447156,0
1149777,276709,0515107662,10
1149778,276721,0590442449,10


## Data Preprocessing
To improve recommendation quality, we apply two thresholds:

- Users: Only users who have rated more than 200 books .

- Books: Only books with at least 50 ratings.

In [ ]:
books.rename(columns={"Book-Title": "title", "Book-Author": "author", "Year-Of-Publication": "year"}, inplace=True)
ratings.rename(columns={"User-ID": "user_id", "Book-Rating": "rating"}, inplace=True)

active_users = ratings['user_id'].value_counts() > 200
user_indices = active_users[active_users].index
ratings = ratings[ratings['user_id'].isin(user_indices)]

rating_with_books = ratings.merge(books, on="ISBN")

rating_counts = rating_with_books.groupby('title')['rating'].count().reset_index()
rating_counts.rename(columns={'rating': 'number_of_ratings'}, inplace=True)

final_rating = rating_with_books.merge(rating_counts, on='title')
final_rating = final_rating[final_rating['number_of_ratings'] >= 50]

final_rating.drop_duplicates(['user_id', 'title'], inplace=True)

In [ ]:
final_rating

,user_id,ISBN,rating,title,author,year,Publisher,Image-URL-S,Image-URL-M,Image-URL-L,number_of_ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,http://images.amazon.com/images/P/002542730X.0...,http://images.amazon.com/images/P/002542730X.0...,82
13,277427,0060930535,0,The Poisonwood Bible: A Novel,Barbara Kingsolver,1999,Perennial,http://images.amazon.com/images/P/0060930535.0...,http://images.amazon.com/images/P/0060930535.0...,http://images.amazon.com/images/P/0060930535.0...,133
15,277427,0060934417,0,Bel Canto: A Novel,Ann Patchett,2002,Perennial,http://images.amazon.com/images/P/0060934417.0...,http://images.amazon.com/images/P/0060934417.0...,http://images.amazon.com/images/P/0060934417.0...,108
18,277427,0061009059,9,One for the Money (Stephanie Plum Novels (Pape...,Janet Evanovich,1995,HarperTorch,http://images.amazon.com/images/P/0061009059.0...,http://images.amazon.com/images/P/0061009059.0...,http://images.amazon.com/images/P/0061009059.0...,108
24,277427,006440188X,0,The Secret Garden,Frances Hodgson Burnett,1998,HarperTrophy,http://images.amazon.com/images/P/006440188X.0...,http://images.amazon.com/images/P/006440188X.0...,http://images.amazon.com/images/P/006440188X.0...,79
...,...,...,...,...,...,...,...,...,...,...,...
487505,275970,1400031354,0,Tears of the Giraffe (No.1 Ladies Detective Ag...,Alexander McCall Smith,2002,Anchor,http://images.amazon.com/images/P/1400031354.0...,http://images.amazon.com/images/P/1400031354.0...,http://images.amazon.com/images/P/1400031354.0...,84
487506,275970,1400031362,0,Morality for Beautiful Girls (No.1 Ladies Dete...,Alexander McCall Smith,2002,Anchor,http://images.amazon.com/images/P/1400031362.0...,http://images.amazon.com/images/P/1400031362.0...,http://images.amazon.com/images/P/1400031362.0...,60
487579,275970,1573229725,0,Fingersmith,Sarah Waters,2002,Riverhead Books,http://images.amazon.com/images/P/1573229725.0...,http://images.amazon.com/images/P/1573229725.0...,http://images.amazon.com/images/P/1573229725.0...,59
487618,275970,1586210661,9,Me Talk Pretty One Day,David Sedaris,2001,Time Warner Audio Major,http://images.amazon.com/images/P/1586210661.0...,http://images.amazon.com/images/P/1586210661.0...,http://images.amazon.com/images/P/1586210661.0...,146


# 3. Matrix Transformation and Model Training

Feature Engineering: Pivot Table to Sparse Matrix
We transform the dataframe into a User-Item Matrix. Since most books are not rated by most users, we use a Compressed Sparse Row (CSR) matrix to save memory.

In [ ]:
book_pivot = final_rating.pivot_table(columns='user_id', index='title', values='rating')
book_pivot.fillna(0, inplace=True)

book_sparse = csr_matrix(book_pivot)

model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_sparse)

print("Model training complete.")

Model training complete.


# 4. Recommendation Engine

Recommendation Function. This function retrieves the $k$ nearest neighbors for a given title. It includes a basic "Fuzzy Matching" logic to suggest correct titles if the input is not found.

In [ ]:
def recommend_book(book_name):
    if book_name not in book_pivot.index:
        print(f"'{book_name}' not found.")
        suggestions = [title for title in book_pivot.index if book_name.lower() in title.lower()]
        if suggestions:
            print(f"Did you mean: '{suggestions[0]}'?")
        return

    book_id = np.where(book_pivot.index == book_name)[0][0]
    distances, suggestions = model.kneighbors(book_pivot.iloc[book_id, :].values.reshape(1, -1), n_neighbors=6)

    print(f"\nSince you liked: {book_name}")
    print("You might also enjoy:")

    for i in range(1, len(suggestions[0])):
        print(f"- {book_pivot.index[suggestions[0][i]]}")



In [ ]:
recommend_book('1984')


Since you liked: 1984
You might also enjoy:
- Animal Farm
- The Catcher in the Rye
- Lord of the Flies
- The Handmaid's Tale
- Slaughterhouse Five or the Children's Crusade: A Duty Dance With Death


# 5. Results and Evaluation

## Model Performance Analysis

The recommendation for "1984" demonstrates that the model effectively captures thematic and cultural similarities between books. The suggestions (Dystopian fiction, classic literature) are highly relevant:
- Animal Farm: Also by George Orwell, sharing similar political themes.
- The Handmaid's Tale: A core dystopian classic.
- Lord of the Flies: Explores similar dark themes of human nature and societal collapse.

